# PINNs-Bernoulli — A Physics-Informed Neural Network, Taught From Scratch

<a href="https://colab.research.google.com/github/harsh147-github/PINNs---Bernoulli/blob/main/notebooks/PINNs_Bernoulli_Tutorial.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

This notebook is the **executable companion to `README.md`**. Every section below pairs the theory (rendered math, exactly as derived in the README) with the *actual, unmodified* source code from this repository (`src/*.py`) — run each cell and watch the numbers move, instead of just reading about them.

**What this repo does, in one sentence:** it trains a small PyTorch network to predict water velocity and pressure through a converging–diverging duct (a venturi) **using only the governing physics equations as the training signal — zero CFD simulation data**, for *any* throat diameter in a design range, all at once.

We will build the whole thing up piece by piece, in the same dependency order the source files use:

1. **Geometry** — the duct shape, pure differentiable arithmetic (`src/geometry.py`)
2. **Analytical solution** — the closed-form exact answer, for grading (`src/analytical.py`)
3. **PINN foundations** — what a neuron computes, forward propagation, autodiff
4. **Network architecture** — the hard-boundary-condition trick (`src/networks.py`)
5. **Physics residuals** — turning a differential equation into a checkable number via `torch.autograd.grad` (`src/physics.py`)
6. **Loss function formulation** — *the centerpiece of this notebook* — turning residuals into the one scalar the optimizer descends, including the automatic loss-weight balancing algorithm (`src/losses.py`)
7. **Collocation sampling** — deciding *where* to check the physics (`src/sampling.py`)
8. **Training loop** — Adam → L-BFGS, run live in this notebook (`src/train.py`)
9. **Evaluation** — grading the trained network against the exact solution, the acceptance gates (`src/evaluate.py`)
10. **Visualization & next steps**

> This notebook trains a small demo model for a few hundred epochs so everything runs in under a minute on Colab's free CPU tier. The full, gate-passing 20,000-epoch run is `scripts/run_stage1_bernoulli.py` — see the last section.


## 0. Setup

Clones the repo (skipped if you're already running this notebook from inside a local checkout) and installs dependencies. Safe to re-run.

In [ ]:
import os, subprocess, sys

IN_REPO = os.path.exists("src/geometry.py")
if not IN_REPO:
    if not os.path.exists("PINNs---Bernoulli"):
        subprocess.run(["git", "clone", "--quiet",
                         "https://github.com/harsh147-github/PINNs---Bernoulli.git"], check=True)
    os.chdir("PINNs---Bernoulli")

print("Working directory:", os.getcwd())


In [ ]:
%pip install -q -r requirements.txt


In [ ]:
import math
import numpy as np
import torch
import matplotlib.pyplot as plt

torch.manual_seed(1234)
np.random.seed(1234)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, "(this notebook runs fine on CPU-only Colab — no GPU required)")


## 1. Geometry — the duct shape (`src/geometry.py`)

The duct is a venturi: same diameter at both ends ($D_{in}$), pinched to a narrower throat ($D_t$) exactly halfway along its length $L$:

$$D(x; D_t) = D_{in} + (D_t - D_{in})\sin^2\!\left(\frac{\pi x}{L}\right), \qquad A(x; D_t) = \frac{\pi}{4}D(x; D_t)^2$$

Check the endpoints: $\sin(0)=\sin(\pi)=0 \Rightarrow D(0)=D(L)=D_{in}$. At the midpoint, $\sin(\pi/2)=1 \Rightarrow D(L/2)=D_t$ — the throat, exactly where we want it.

**Why `torch`, not `numpy`, inside this function?** Every function below gets called later with `x.requires_grad_(True)` (Section 5). PyTorch can only differentiate through operations it recorded, so the geometry law has to be built from `torch.sin`, not plain Python floats — this is what lets it later become part of a differentiable chain.

**Non-dimensionalization** (why the network never sees raw metres): $\tilde x = x/L \in [0,1]$, $\tilde D_t = D_t/D_{in} \in [0.4, 0.9]$, $\tilde A = A/A_{in}$. Neural networks train well when inputs and outputs sit around order 1 — six-orders-of-magnitude gaps (like $x \sim 1\,\text{m}$ vs $p \sim 10^5\,\text{Pa}$) badly distort the loss landscape.

Here is the **exact, unmodified code** from `src/geometry.py`:

In [ ]:
from src import geometry

import inspect
print(inspect.getsource(geometry.diameter))
print(inspect.getsource(geometry.area_nondim))


In [ ]:
# Demo: plot the duct shape for three throat diameters
L, D_in = 3.0, 0.5
x = torch.linspace(0.0, L, 300).reshape(-1, 1)

fig, ax = plt.subplots(figsize=(8, 3))
for Dt in [0.20, 0.30, 0.45]:
    dt = torch.full_like(x, Dt)
    D = geometry.diameter(x, dt, L, D_in)
    ax.plot(x.numpy(), (D / 2).numpy(), label=f"$D_t$={Dt} m")
    ax.plot(x.numpy(), (-D / 2).numpy(), color=ax.lines[-1].get_color())
ax.axvline(L / 2, color="gray", ls=":", lw=1, label="throat ($x=L/2$)")
ax.set(xlabel="x [m]", ylabel="duct half-width [m]", title="Venturi geometry at three throat diameters")
ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.show()


## 2. The analytical solution — ground truth for grading (`src/analytical.py`)

Two physical laws pin the flow down completely for an incompressible, inviscid fluid:

- **Continuity** (mass in = mass out): $A(x)V(x) = A_{in}V_{in} \Rightarrow V(x) = \dfrac{V_{in}A_{in}}{A(x)}$
- **Bernoulli's theorem**: $p + \tfrac12\rho V^2 = p_{in} + \tfrac12\rho V_{in}^2 \Rightarrow p(x) = p_{in} + \tfrac12\rho\big(V_{in}^2 - V(x)^2\big)$

In the non-dimensional variables ($\tilde x = x/L$, $\tilde V = V/V_{in}$, $\tilde p = (p-p_{in})/(\tfrac12\rho V_{in}^2)$), every constant cancels and these collapse to two lines:

$$\tilde V = \frac{1}{\tilde A(\tilde x; \tilde D_t)}, \qquad \tilde p = 1 - \tilde V^2$$

This is the **only place** the correct answer is ever computed in the entire repo — and it is *never* shown to the network during training. It exists purely so we can grade the trained network honestly afterward.

In [ ]:
from src import analytical

print(inspect.getsource(analytical.velocity_exact_nondim))
print(inspect.getsource(analytical.pressure_exact_nondim))


In [ ]:
# Demo: the exact solution across the duct, for several throat diameters
xt = torch.linspace(0.0, 1.0, 300).reshape(-1, 1)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
for Dt_tilde in [0.4, 0.6, 0.9]:
    dtt = torch.full_like(xt, Dt_tilde)
    v = analytical.velocity_exact_nondim(xt, dtt)
    p = analytical.pressure_exact_nondim(xt, dtt)
    axes[0].plot(xt.numpy(), v.numpy(), label=f"$\\tilde D_t$={Dt_tilde}")
    axes[1].plot(xt.numpy(), p.numpy(), label=f"$\\tilde D_t$={Dt_tilde}")
axes[0].set(xlabel="$\\tilde x$", ylabel="$\\tilde V$", title="Exact velocity")
axes[1].set(xlabel="$\\tilde x$", ylabel="$\\tilde p$", title="Exact pressure")
for ax in axes:
    ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print("Note: smaller throat -> higher peak velocity -> deeper pressure drop (Bernoulli trade-off).")


## 3. PINN foundations, from absolute zero

### 3.1 What a single neuron computes

Forget "neural network" for a second — one artificial neuron is arithmetic you could do on paper:

$$z = w_1 x_1 + w_2 x_2 + b, \qquad \text{output} = \phi(z)$$

multiply each input by its own weight, add a bias, squash through a nonlinear activation $\phi$.

### 3.2 Why `tanh`, not `ReLU`

Section 5 differentiates this network's *output* with respect to its *input* — and the loss built from that residual gets differentiated **again** with respect to the network's *weights* during backprop. `tanh` is smooth to every order everywhere; `ReLU` has a sharp corner at zero where its second derivative doesn't exist. For an ordinary classifier that corner is harmless — here it would poison the physics residual itself, since the residual is literally built from a derivative of the network's output.

### 3.3 Automatic differentiation — the one trick that makes this all work

We never write down $dV/dx$ by hand, and we never approximate it with finite differences. PyTorch records every operation during the forward pass (a "computation graph"), and `torch.autograd.grad` walks that graph backward applying the chain rule *exactly*. Let's see it work on something we can check by hand: $y = x^3$, so $dy/dx = 3x^2$.

In [ ]:
x = torch.tensor([2.0], requires_grad=True)
y = x ** 3
dy_dx = torch.autograd.grad(y, x, create_graph=True)[0]
print(f"autograd:  dy/dx at x=2 -> {dy_dx.item()}")
print(f"by hand:   3*x^2 at x=2  -> {3 * 2.0**2}")


Exact match, not an approximation. This is the entire mechanism Section 5 uses to turn "does the network's output satisfy this differential equation" into a number.

## 4. Network architecture — the hard-boundary-condition trick (`src/networks.py`)

| Choice | Value | Why |
|---|---|---|
| Inputs | $(\tilde x, \tilde D_t)$ | position + design parameter → *parametric* surrogate |
| Outputs | $(\hat{\tilde V}, \hat{\tilde p})$ | the two flow variables |
| Hidden layers | 6 × 64 neurons, `tanh` | smooth, fits 1D solutions comfortably |
| Init | Xavier/Glorot uniform | keeps activation variance stable at depth |
| Precision | float64 | PDE residuals amplify round-off |

**The inlet boundary condition** $\tilde V(0)=1,\ \tilde p(0)=0$ could be added as a loss penalty ("please be close to this at $x=0$") — but a penalty is a *suggestion* the optimizer can trade off against other goals. Instead the output transform below bakes the condition into the algebra itself:

$$\hat{\tilde V}(\tilde x) = 1 + \tilde x \cdot N_V(\tilde x, \tilde D_t), \qquad \hat{\tilde p}(\tilde x) = \tilde x \cdot N_p(\tilde x, \tilde D_t)$$

No matter what the raw network $N_V, N_p$ outputs, at $\tilde x = 0$ the $\tilde x \cdot (\cdot)$ term is exactly zero. There is no combination of weights that can violate the inlet condition — it's guaranteed by the *shape of the function*, not by training.

In [ ]:
from src import networks

print(inspect.getsource(networks.PINN.forward))


In [ ]:
model = networks.PINN(n_hidden_layers=6, n_neurons=64, activation="tanh", hard_bc=True).double()
print(model)
print("trainable parameters:", model.count_parameters())

# Verify the hard BC holds EXACTLY, even on this untrained, randomly-initialized network:
xt0 = torch.zeros(5, 1, dtype=torch.float64)
dtt0 = torch.rand(5, 1, dtype=torch.float64) * 0.5 + 0.4
v0, p0 = model(xt0, dtt0)
print("\nV~(0) for 5 random throat diameters (should be exactly 1.0):", v0.flatten().tolist())
print("p~(0) for 5 random throat diameters (should be exactly 0.0):", p0.flatten().tolist())


## 5. Physics residuals — the core trick (`src/physics.py`)

This is where "neural network" and "differential equation" actually meet.

Continuity says $\dfrac{d}{d\tilde x}\big[\tilde A(\tilde x)\tilde V(\tilde x)\big] = 0$ — the *slope* of that product must be zero everywhere. To check that on the network's current guess, we need $d\tilde V/d\tilde x$ of that guess — and since the guess is a big pile of matrix multiplies and `tanh` calls, "what's the slope of the output of a big pile of arithmetic" is exactly what `torch.autograd.grad` computes.

$$r_{cont} = \frac{d}{d\tilde x}\Big[\tilde A(\tilde x, \tilde D_t)\cdot\hat{\tilde V}\Big], \qquad r_{mom} = \frac{d}{d\tilde x}\Big[\hat{\tilde p} + \hat{\tilde V}^2\Big]$$

($r_{mom}$ is the differential form of Bernoulli's theorem: total head $\tilde p + \tilde V^2$ must be *constant*, i.e. its slope is zero.)

**`create_graph=True`** — the one flag that would silently break everything if forgotten. We are not done differentiating once we have the residual: `losses.py` squares it, and `train.py` later calls `.backward()` on the *total loss* to get the gradient with respect to the network's *weights* — a **second** differentiation, one level up. `create_graph=True` keeps this first derivative itself differentiable, so that second backward pass has something to walk through.

In [ ]:
from src import physics

print(inspect.getsource(physics._grad))
print(inspect.getsource(physics.residual_continuity))
print(inspect.getsource(physics.residual_momentum))


In [ ]:
# Sanity check #1: an UNTRAINED (random) network should badly violate both equations.
xt_demo = torch.rand(2000, 1, dtype=torch.float64)
dtt_demo = torch.rand(2000, 1, dtype=torch.float64) * 0.5 + 0.4

r_cont_untrained = physics.residual_continuity(model, xt_demo, dtt_demo)
r_mom_untrained = physics.residual_momentum(model, xt_demo, dtt_demo)
print("untrained network:")
print(f"  mean|r_cont| = {r_cont_untrained.abs().mean().item():.4f}   (should be far from 0)")
print(f"  mean|r_mom|  = {r_mom_untrained.abs().mean().item():.4f}   (should be far from 0)")


In [ ]:
# Sanity check #2: wrap the EXACT analytical solution as a fake "model" and check
# its residual is zero to machine precision -- this is exactly what tests/test_analytical.py checks.
class ExactAsModel:
    def __call__(self, xt, dtt):
        return analytical.velocity_exact_nondim(xt, dtt), analytical.pressure_exact_nondim(xt, dtt)

exact_model = ExactAsModel()
r_cont_exact = physics.residual_continuity(exact_model, xt_demo.clone(), dtt_demo)
r_mom_exact = physics.residual_momentum(exact_model, xt_demo.clone(), dtt_demo)
print("exact analytical solution, fed through the SAME residual code:")
print(f"  max|r_cont| = {r_cont_exact.abs().max().item():.2e}   (machine precision)")
print(f"  max|r_mom|  = {r_mom_exact.abs().max().item():.2e}   (machine precision)")


This is the whole point of the PINN approach: the residual functions are *correct* (they collapse to ~0 on the true solution), and training is nothing more than nudging the network's weights until its own residuals shrink toward that same zero — with no CFD data involved anywhere.

## 6. Loss function formulation — turning residuals into the number we optimize (`src/losses.py`)

This is the section you asked to see spelled out in full: **how do we go from "the network violates continuity by some amount at 4,096 scattered points" to the single scalar that `loss.backward()` needs?**

### 6.1 From a residual to a loss term

A residual isn't a loss yet: it's signed (can be positive or negative) and it's a *vector* of thousands of per-point numbers, not one number. Two operations fix both problems at once:

$$L_{cont} = \operatorname{mean}\big(r_{cont}^2\big), \qquad L_{mom} = \operatorname{mean}\big(r_{mom}^2\big)$$

- **Squaring** makes a residual of $-0.3$ count exactly as bad as $+0.3$ (direction doesn't matter, only size), and punishes large violations much harder than small ones ($0.3^2=0.09$ vs $0.03^2=0.0009$ — a 10× bigger error costs **100×** more loss). This pushes training to stamp out the worst offenders first.
- **Averaging** ("mean") over every collocation point turns "thousands of per-point numbers" into the single scalar the optimizer needs.

### 6.2 The total loss

$$L = \lambda_{cont}\, L_{cont} + \lambda_{mom}\, L_{mom}\ \big(+\ \lambda_{bc}\, L_{bc}\ \text{only if BCs are soft, not hard-wired}\big)$$

Let's build this from scratch, by hand, exactly the way `losses.py` does it — one line at a time:

In [ ]:
from src import losses

print(inspect.getsource(losses.loss_continuity))
print(inspect.getsource(losses.loss_momentum))


In [ ]:
# Building L_cont from scratch, step by step, on our untrained model:
# step 1: the residual (a vector, one number per collocation point)
r = physics.residual_continuity(model, xt_demo, dtt_demo)
print("residual vector shape:", tuple(r.shape), "  (one r_cont per collocation point)")

# step 2: square it (direction stops mattering, big violations dominate)
r_squared = r ** 2

# step 3: average -> ONE scalar number
L_cont_manual = r_squared.mean()
print("L_cont, built by hand:  ", L_cont_manual.item())

# compare against the actual library function -- should match exactly
L_cont_lib = losses.loss_continuity(model, xt_demo, dtt_demo)
print("L_cont, from losses.py: ", L_cont_lib.item())


### 6.3 Why the $\lambda$ weights exist at all — a gradient-pathology demo

Continuity and momentum don't naturally produce gradients of the same size. With $\lambda_{cont}=\lambda_{mom}=1$ fixed forever, the optimizer will happily drive whichever term has the *louder* gradient toward zero while quietly neglecting the other (Wang, Teng & Perdikaris 2021 — a "gradient pathology"). Let's actually measure this on our network, right now:

In [ ]:
params = [p for p in model.parameters() if p.requires_grad]

l_cont = losses.loss_continuity(model, xt_demo, dtt_demo)
l_mom = losses.loss_momentum(model, xt_demo, dtt_demo)

g_cont = torch.autograd.grad(l_cont, params, retain_graph=True, allow_unused=True)
g_mom = torch.autograd.grad(l_mom, params, retain_graph=True, allow_unused=True)

mean_abs_grad_cont = torch.cat([g.abs().flatten() for g in g_cont if g is not None]).mean().item()
mean_abs_grad_mom = torch.cat([g.abs().flatten() for g in g_mom if g is not None]).mean().item()

print(f"mean|grad L_cont| w.r.t. weights = {mean_abs_grad_cont:.3e}")
print(f"mean|grad L_mom|  w.r.t. weights = {mean_abs_grad_mom:.3e}")
print(f"ratio = {max(mean_abs_grad_cont, mean_abs_grad_mom) / min(mean_abs_grad_cont, mean_abs_grad_mom):.1f}x")
print("\nWith lambda=1 for both, the optimizer's step is dominated by whichever term's gradient is larger above.")


### 6.4 The fix: LR-annealing (Wang et al. 2021, Algorithm 1)

Every `anneal_every` epochs, re-measure each term's gradient *magnitude* and rebalance so no term dominates:

$$\hat\lambda_i = \frac{\max_j |\nabla L_j|}{\operatorname{mean}|\nabla L_i|}, \qquad \lambda_i \leftarrow (1-\alpha)\lambda_i + \alpha\hat\lambda_i \quad (\alpha = 0.9)$$

The update leans 90% toward the fresh estimate each time it fires — this is `maybe_anneal()` below, verbatim:

In [ ]:
print(inspect.getsource(losses.LossWeights.maybe_anneal))


In [ ]:
# Run the annealing update by hand a few times on our (untrained) model, and watch
# lambda_cont / lambda_mom move to compensate for the gradient-magnitude gap we just measured.
weights = losses.LossWeights(weighting="annealing", anneal_alpha=0.9, anneal_every=1, anneal_warmup=0)
print(f"{'step':>4} | {'lambda_cont':>12} | {'lambda_mom':>12}")
for step in range(5):
    total, parts = weights.total(model, xt_demo, dtt_demo, hard_bc=True)
    weights.maybe_anneal(step, model, parts)   # <-- BEFORE backward(), see next cell
    print(f"{step:>4} | {weights.values['cont']:>12.3f} | {weights.values['mom']:>12.3f}")


### 6.5 The one ordering bug that would silently break annealing

Look at `train.py`'s loop: `weights.maybe_anneal(...)` runs **before** `total.backward()`, never after. This is deliberate, not arbitrary: `maybe_anneal` needs to call `torch.autograd.grad` on each *individual* loss term to measure its gradient size, and that only works while the computation graph is still alive. `.backward()` frees that graph once it runs — swap the order and annealing silently breaks (this bug existed in an earlier version of this repo and was fixed; see `CLAUDE.md` §5).

### 6.6 Assembling the total loss — `LossWeights.total()`

In [ ]:
print(inspect.getsource(losses.LossWeights.total))


That's the entire loss formulation: **residual → square → mean → weighted sum**, with the weights themselves re-measured periodically from the loss landscape so neither equation gets neglected. Nothing here ever compares the network's output to a "correct answer" — the only signal is how well the network's own output satisfies the governing equations.

## 7. Collocation sampling — deciding *where* to check the physics (`src/sampling.py`)

The residuals above can only ever be checked at specific points — never at literally every point in a continuous duct. **Latin Hypercube Sampling (LHS)** draws points over the 2D box $[0,1]\times[\tilde D_{t,min}, \tilde D_{t,max}]$ so they're spread evenly across both axes (no clumping like pure-random, no rigid repeats like a grid) while still varying every draw. Position *and* throat diameter are sampled together, every time — this is the literal mechanical reason one trained network becomes a **parametric surrogate** instead of a solver for one fixed geometry.

In [ ]:
from src import sampling

print(inspect.getsource(sampling.lhs_collocation))


In [ ]:
xt_lhs, dtt_lhs = sampling.lhs_collocation(n_points=512 * 8, dtt_min=0.4, dtt_max=0.9, seed=1234)

fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(xt_lhs.numpy(), dtt_lhs.numpy(), s=3, alpha=0.4)
ax.set(xlabel="$\\tilde x$", ylabel="$\\tilde D_t$", title=f"{xt_lhs.shape[0]} LHS collocation points")
ax.grid(alpha=0.3)
plt.show()


## 8. Training loop — Adam then L-BFGS (`src/train.py`), run live

The full protocol (`scripts/run_stage1_bernoulli.py`, `configs/default.yaml`): **20,000 Adam epochs** (lr $10^{-3}$, decayed ×0.95 every 2,000 epochs) followed by **5,000 L-BFGS iterations** (strong-Wolfe line search) for a final polish. Collocation points are resampled every 1,000 epochs (Section 7) so the network can't quietly overfit to one fixed point set.

To keep this notebook fast on Colab's free CPU tier, we'll run a **short demo** (a few hundred epochs) using the exact same functions imported from `src/`, not a reimplementation. The full run is one command — see Section 10.

In [ ]:
import yaml

with open("configs/default.yaml") as f:
    cfg = yaml.safe_load(f)

torch.manual_seed(cfg["seed"])
demo_model = networks.PINN(
    n_hidden_layers=cfg["network"]["n_hidden_layers"],
    n_neurons=cfg["network"]["n_neurons"],
    activation=cfg["network"]["activation"],
    hard_bc=cfg["network"]["hard_bc"],
).double()

g = cfg["geometry"]
dtt_min, dtt_max = g["dt_min"] / g["D_in"], g["dt_max"] / g["D_in"]

demo_weights = losses.LossWeights(
    weighting=cfg["loss"]["weighting"],
    anneal_alpha=cfg["loss"]["anneal_alpha"],
    anneal_every=cfg["loss"]["anneal_every"],
    anneal_warmup=cfg["loss"]["anneal_warmup"],
    values={"cont": cfg["loss"]["lambda_cont"], "mom": cfg["loss"]["lambda_mom"], "bc": cfg["loss"]["lambda_bc"]},
)

opt = torch.optim.Adam(demo_model.parameters(), lr=cfg["training"]["adam_lr"])
DEMO_EPOCHS = 400  # the real run uses cfg["training"]["adam_epochs"] = 20000

xt_f, dtt_f = sampling.lhs_collocation(
    cfg["training"]["n_collocation_x"] * cfg["training"]["n_dt_values"], dtt_min, dtt_max, cfg["seed"]
)

loss_history = []
for epoch in range(1, DEMO_EPOCHS + 1):
    opt.zero_grad()
    total, parts = demo_weights.total(demo_model, xt_f, dtt_f, hard_bc=True)
    demo_weights.maybe_anneal(epoch, demo_model, parts)  # BEFORE backward -- Section 6.5
    total.backward()
    opt.step()
    loss_history.append(total.item())
    if epoch % 100 == 0:
        print(f"epoch {epoch:5d} | total loss = {total.item():.3e} | "
              f"lambda_cont={demo_weights.values['cont']:.2f}  lambda_mom={demo_weights.values['mom']:.2f}")

print("\ndemo training done (this is NOT the full 20,000-epoch run, so gates below will NOT pass yet).")


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.semilogy(loss_history)
ax.set(xlabel="epoch", ylabel="total loss (log scale)", title=f"Demo training loss ({DEMO_EPOCHS} Adam epochs)")
ax.grid(alpha=0.3, which="both")
plt.show()


## 9. Evaluation — grading against the exact solution (`src/evaluate.py`)

$$\text{rel-}L^2 = \frac{\lVert \text{prediction} - \text{exact}\rVert}{\lVert \text{exact}\rVert}$$

Crucially, this is measured **only on held-out throat diameters** (0.225, 0.275, 0.325, 0.375, 0.425 m) that never once appeared in any `lhs_collocation` draw. A network that merely memorized its training geometries would fail here — one that actually learned continuity and Bernoulli in general does not care that it's being asked about an unseen $D_t$, because the equations hold everywhere.

**The three acceptance gates** (`CLAUDE.md` §5 / README §9): rel-$L^2(\tilde V) < 10^{-3}$, rel-$L^2(\tilde p) < 10^{-2}$, and the Bernoulli-invariant check $|\hat{\tilde p}+\hat{\tilde V}^2-1| < 1\%$ everywhere.

In [ ]:
from src import evaluate

print(inspect.getsource(evaluate.validate_held_out))


In [ ]:
report = evaluate.full_report(demo_model, cfg)
print(f"rel-L2(V~)            = {report['rel_l2_V']:.3e}   (gate < {cfg['acceptance']['rel_l2_V']:.0e})")
print(f"rel-L2(p~)            = {report['rel_l2_p']:.3e}   (gate < {cfg['acceptance']['rel_l2_p']:.0e})")
print(f"max|p~+V~^2-1|        = {report['total_head_max_err']:.3e}   (gate < {cfg['acceptance']['total_head_tol']:.0e})")
print(f"ALL GATES PASSED?     = {report['all_passed']}")
print("\n(Expected: FALSE after only ~400 demo epochs. The committed reference run — 20,000 Adam")
print(" epochs + 5,000 L-BFGS iterations, `scripts/run_stage1_bernoulli.py` — passes all three")
print(" with roughly a 10x margin: rel-L2(V)=1.03e-4, rel-L2(p)=2.33e-4, max error=3.4e-4.)")


## 10. Visualization: demo network vs exact solution

Even after only a few hundred epochs, the demo network should already be visibly tracking the exact curve, just not yet within the tight acceptance tolerance.

In [ ]:
xt_plot = torch.linspace(0, 1, 300, dtype=torch.float64).reshape(-1, 1)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
for Dt_tilde in [0.4, 0.65, 0.9]:
    dtt_plot = torch.full_like(xt_plot, Dt_tilde)
    with torch.no_grad():
        v_pred, p_pred = demo_model(xt_plot, dtt_plot)
    v_exact = analytical.velocity_exact_nondim(xt_plot, dtt_plot)
    p_exact = analytical.pressure_exact_nondim(xt_plot, dtt_plot)

    (l,) = axes[0].plot(xt_plot.numpy(), v_pred.numpy(), label=f"$\\tilde D_t$={Dt_tilde}")
    axes[0].plot(xt_plot.numpy(), v_exact.numpy(), "--", color=l.get_color(), alpha=0.6)
    (l,) = axes[1].plot(xt_plot.numpy(), p_pred.numpy(), label=f"$\\tilde D_t$={Dt_tilde}")
    axes[1].plot(xt_plot.numpy(), p_exact.numpy(), "--", color=l.get_color(), alpha=0.6)

axes[0].set(xlabel="$\\tilde x$", ylabel="$\\tilde V$", title="Velocity: demo net (solid) vs exact (dashed)")
axes[1].set(xlabel="$\\tilde x$", ylabel="$\\tilde p$", title="Pressure: demo net (solid) vs exact (dashed)")
for ax in axes:
    ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()


## 11. Next steps: the full, gate-passing pipeline

Everything above used the exact functions this repository trains with — just for fewer epochs, to keep the notebook fast. To reproduce the committed, gate-passing result (and get the ParaView `.vtu` time series + MATLAB `.mat` export), run from a full clone:

```bash
git clone https://github.com/harsh147-github/PINNs---Bernoulli.git
cd PINNs---Bernoulli
pip install -r requirements.txt
pytest tests/ -q                                                    # 6 tests must pass
python scripts/run_stage1_bernoulli.py --config configs/default.yaml # ~15-20 min on CPU
```

This trains for the full 20,000 Adam epochs + 5,000 L-BFGS iterations, evaluates all three acceptance gates, and writes:

- `results/figures/` — the 5 PNGs referenced throughout `README.md` §10
- `results/paraview/` — 25 `.vtu` files (open together in ParaView, color by `p` or `V`, press Play to animate the throat sweep)
- `results/matlab/nozzle_surrogate.mat` — load with `postprocessing/matlab/load_and_plot.m`, no toolboxes required

**Read the rest of the theory** in `README.md` — this notebook mirrors its Sections 1–10 exactly, and `CLAUDE.md` documents the binding build/run contract this repo follows.

**Watch training happen live** (`configs/default.yaml`'s `training.live_dashboard` / `live_network_viz` flags, `src/netviz.py`, `scripts/make_learning_replay.py`) — see `README.md` / `CLAUDE.md` §3 for the dashboard and network-diagram visualizers this repo also includes.
